**© Copyright AIDENTIFY. All rights reserved.**

# Part 5 | Session 04: Tool Calling (Function Calling)과 Agent 루프

## 🎯 개요

Session 03에서 Claude Code가 파일을 읽고 명령을 실행하며 스스로 검증하는 모습을 봤습니다. 그 동작의 바탕에는 **Tool Calling**(도구 호출)이 있습니다.
LLM이 외부 도구(함수, API)를 호출해 실시간 정보를 가져오거나 작업을 수행하게 하는 기능입니다.

이번 세션에서는 이 메커니즘을 OpenAI API로 **바닥부터** 구현합니다. 도구 정의 → 첫 호출 → 1회 왕복 파이프라인 → 멀티 도구 → **Agent 루프** 순서로 쌓아 올립니다.

### 왜 Tool Calling이 필요한가?

| LLM의 한계 | Tool Calling 해결책 |
|-----------|--------------------|
| 실시간 정보 없음 | 날씨 API, 뉴스 API 호출 |
| 정확한 계산 불가 | 계산기 도구 호출 |
| 외부 시스템 접근 불가 | DB 조회, 이메일 발송 등 |
| 최신 데이터 부족 | 검색 엔진 호출 |

### 이번 세션의 구성

```
1️⃣ 환경 준비
2️⃣ 도구 정의 (JSON Schema)          — 모델에게 "이런 도구가 있다"고 알리기
3️⃣ 첫 호출                          — 모델은 실행하지 않고 tool_calls 로 "요청"만 한다
4️⃣ 도구 구현 + 1회 왕복 파이프라인    — 선택 → 실행 → 응답
5️⃣ 멀티 도구 호출                    — 한 번에 여러 도구
6️⃣ Agent 루프                       — 도구 결과를 보고 다시 판단, 끝날 때까지 반복
7️⃣ 실전 도구 확장
```

### 학습 목표

- ✅ Tool Calling의 개념과 동작 원리 이해
- ✅ 도구 정의 (JSON Schema) 작성법과 `description` 의 중요성
- ✅ 1회 왕복 파이프라인과 멀티 도구 호출 구현
- ✅ Agent 루프 구현 — 1회 왕복과 무엇이 다른지 이해

### 실습 환경

- 🔧 GPU 불필요 (API 기반 실습)
- 🔧 OpenAI API 키 필요 (gpt-4o-mini 사용) — 노트북과 같은 폴더의 `.env` 에 저장 (Session 03 참고)

## 1️⃣ 환경 준비

In [ ]:
# 필수 라이브러리 & OpenAI 클라이언트 — 키는 같은 폴더의 .env 에서 로드 (Session 03 과 동일한 방식)
import os, json, random
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
assert os.environ.get("OPENAI_API_KEY", "").startswith("sk-"), \
    "OPENAI_API_KEY 없음 — 노트북과 같은 폴더의 .env 에 OPENAI_API_KEY=sk-... 를 넣으세요"

client = OpenAI()          # 환경 변수의 OPENAI_API_KEY 를 자동 사용
MODEL = "gpt-4o-mini"      # 비용 효율적 모델

print(f"✅ OpenAI 클라이언트 초기화 완료 — 모델: {MODEL}")

## 2️⃣ 도구 정의 (JSON Schema) — 모델에게 도구를 "설명"하기

도구는 **JSON Schema** 형식으로 정의합니다. 모델은 이 스키마를 보고 어떤 도구를 호출할지, 어떤 인자를 넘길지 결정합니다.

> ⚠️ 여기서 정의하는 것은 **설명서**일 뿐 실제 함수가 아닙니다. 실제 Python 함수는 4️⃣에서 따로 구현하고 이름으로 연결합니다.
> 이 분리 덕분에 Session 05에서 같은 설명서를 MCP 서버로 옮길 수 있습니다.

이 노트북 전체에서 `get_weather`, `calculate`, `search_web` 세 도구를 사용합니다.

In [ ]:
# 이 노트북에서 사용할 도구 3개 정의 (설명서)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "주어진 도시의 현재 날씨 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "도시 이름 (예: 서울, 부산, 제주)"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "수학 표현식을 계산합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "계산할 수학 표현식 (예: 2 + 3 * 4)"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "웹에서 정보를 검색합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "검색 쿼리"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("정의된 도구 목록:")
for tool in tools:
    func = tool["function"]
    params = list(func["parameters"]["properties"].keys())
    print(f"\n  🔧 {func['name']}")
    print(f"     설명: {func['description']}")
    print(f"     파라미터: {params}")

In [ ]:
# JSON Schema 상세 구조 설명
print("\n📋 JSON Schema 구조 설명")
print("="*60)

example_schema = {
    "type": "function",                     # 고정값
    "function": {
        "name": "함수_이름",                  # 모델이 호출할 함수명
        "description": "함수 설명 (중요!)",   # 모델이 도구 선택에 사용
        "parameters": {                      # 함수 파라미터 정의
            "type": "object",                # 고정값
            "properties": {                  # 각 파라미터 정의
                "param1": {
                    "type": "string",         # string, number, boolean, array 등
                    "description": "파라미터 설명"
                },
                "param2": {
                    "type": "number",
                    "description": "숫자 파라미터"
                }
            },
            "required": ["param1"]           # 필수 파라미터 목록
        }
    }
}

print(json.dumps(example_schema, indent=2, ensure_ascii=False))
print("\n📌 'description'이 매우 중요합니다 - 모델이 이를 보고 도구를 선택합니다!")

## 3️⃣ 첫 호출 — 모델은 도구를 실행하지 않고 "요청"만 한다

도구 정의를 `tools` 파라미터로 넘기고 질문하면, 모델은 함수를 실행하는 대신 **"이 도구를 이 인자로 불러 달라"** 는 `tool_calls` 를 돌려줍니다.
실행은 전적으로 **우리 코드**의 몫입니다. 이것이 Tool Calling 의 가장 중요한 사실입니다.

```
┌──────────────────────────────────────────────────┐
│  1. 사용자 질문        "서울 날씨 알려줘"            │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  2. LLM 판단 (도구 선택)                           │
│     - 도구 목록의 description 을 보고 적합한 도구 선택 │
│     - 인자(arguments) 추출                         │
│     → tool_calls: get_weather(city="서울")   ← 여기까지가 모델의 일 │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  3. 도구 실행 (우리 코드)                           │
│     → {"temperature": 15, "condition": "맑음"}    │
└──────────────┬───────────────────────────────────┘
               ▼
┌──────────────────────────────────────────────────┐
│  4. LLM 최종 응답 생성 (결과를 role="tool" 로 전달)  │
│     → "서울은 현재 15도이고 맑습니다."               │
└──────────────────────────────────────────────────┘
```

아래 셀은 2단계까지만 실행해 응답 객체를 관찰합니다.

In [ ]:
# 첫 호출: 도구 정의만 넘기고 질문 → 모델은 실행 대신 tool_calls 를 돌려준다
for question in ["서울 날씨 어때?", "안녕하세요! 반갑습니다."]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",   # 도구 사용 여부를 모델이 판단
    )
    message = response.choices[0].message

    print(f"👤 {question}")
    print(f"   content   : {message.content}")
    if message.tool_calls:
        for tc in message.tool_calls:
            print(f"   tool_calls: {tc.function.name}({tc.function.arguments})   id={tc.id}")
    else:
        print("   tool_calls: None   → 도구 없이 바로 답변")
    print()

### 모델의 도구 선택 기준

| 질문 | 도구 선택 | 이유 |
|------|-----------|------|
| 서울 날씨 알려줘 | ✅ get_weather | 실시간 정보 필요 |
| 123 * 456은? | ✅ calculate | 정확한 계산 필요 |
| 최신 AI 뉴스 찾아줘 | ✅ search_web | 외부 정보 검색 필요 |
| 안녕하세요! | ❌ 도구 불필요 | 일반 대화 |
| 파이썬이 뭐야? | ❌ 도구 불필요 | 내장 지식으로 답변 가능 |
| 서울과 부산 날씨 비교 | ✅ get_weather × 2 | 멀티 도구 호출 (5️⃣) |

> 📌 판단 근거는 오직 **도구의 `description`** 입니다. 설명이 모호하면 엉뚱한 도구를 고르거나 필요한 도구를 건너뜁니다.

## 4️⃣ 도구 구현과 1회 왕복 파이프라인 (선택 → 실행 → 응답)

이제 3️⃣의 3~4단계를 채웁니다. 2️⃣의 설명서와 같은 이름의 Python 함수를 구현하고, `tool_calls` 를 받아 실행한 뒤 결과를 `role="tool"` 메시지로 돌려줘 최종 응답을 받습니다.

In [ ]:
# 도구 함수 구현 (실제 API 대신 시뮬레이션)
import random

def get_weather(city: str) -> dict:
    """날씨 API 시뮬레이션"""
    weathers = ["맑음", "흐림", "비", "눈", "구름 많음"]
    result = {
        "city": city,
        "temperature": random.randint(0, 35),
        "condition": random.choice(weathers),
        "humidity": random.randint(30, 90)
    }
    print(f"  🌤️ [도구 실행] get_weather(city='{city}') → {result}")
    return result

def calculate(expression: str) -> dict:
    """계산기 도구"""
    try:
        result = eval(expression)  # 실습용 (보안상 eval은 실무에서 주의)
        print(f"  🔢 [도구 실행] calculate(expression='{expression}') → {result}")
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"error": str(e)}

def search_web(query: str) -> dict:
    """검색 시뮬레이션"""
    results = {
        "query": query,
        "results": [
            {"title": f"{query} 관련 최신 정보", "snippet": f"{query}에 대한 검색 결과입니다."},
            {"title": f"{query} 가이드", "snippet": f"{query}를 이해하기 위한 종합 가이드입니다."}
        ]
    }
    print(f"  🔍 [도구 실행] search_web(query='{query}')")
    return results

# 도구 이름 → 함수 매핑
available_functions = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_web": search_web
}

print("✅ 도구 함수 3개 정의 완료")
print(f"📌 사용 가능한 도구: {list(available_functions.keys())}")

In [ ]:
# 1회 왕복 파이프라인: 판단 → (도구 실행) → 최종 응답. 도구 호출은 최대 한 라운드만 처리한다
def run_tool_calling(user_message, tools, available_functions, model=MODEL):
    """Tool Calling 전체 파이프라인 실행"""
    print(f"\n{'='*60}")
    print(f"👤 사용자: {user_message}")
    
    messages = [{"role": "user", "content": user_message}]
    
    # Step 1: 모델에게 질문 (도구 사용 여부 판단)
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    # Step 2: 도구 호출이 필요한지 확인
    if assistant_message.tool_calls:
        print(f"\n🤖 모델 판단: 도구 호출 필요!")
        messages.append(assistant_message)
        
        # Step 3: 각 도구 실행
        for tool_call in assistant_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"\n  📞 도구 호출: {func_name}({func_args})")
            
            # 도구 실행
            if func_name in available_functions:
                result = available_functions[func_name](**func_args)
            else:
                result = {"error": f"Unknown function: {func_name}"}
            
            # 도구 결과를 메시지에 추가
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result, ensure_ascii=False)
            })
        
        # Step 4: 도구 결과로 최종 응답 생성
        final_response = client.chat.completions.create(
            model=model,
            messages=messages
        )
        final_answer = final_response.choices[0].message.content
    else:
        print(f"\n🤖 모델 판단: 도구 필요 없음 (직접 응답)")
        final_answer = assistant_message.content
    
    print(f"\n🤖 최종 응답: {final_answer}")
    return final_answer

print("✅ Tool Calling 파이프라인 함수 정의 완료")

In [ ]:
# 도구가 필요한 질문들 테스트
print("📊 Tool Calling 테스트")
print("="*60)

test_questions = [
    "서울 날씨 알려줘",
    "1234 곱하기 5678은?",
    "LLM 파인튜닝에 대해 검색해줘",
    "안녕하세요! 반갑습니다."  # 도구 불필요한 질문
]

for q in test_questions:
    run_tool_calling(q, tools, available_functions)

## 5️⃣ 멀티 도구 호출

모델이 하나의 요청에서 **여러 도구를 동시에 호출**할 수 있습니다. `tool_calls` 배열에 여러 항목이 담겨 오고, 각 결과를 `tool_call_id` 로 짝지어 돌려줍니다.

> 동시 호출은 도구들이 **서로 독립적일 때** 가능합니다. 앞 도구의 결과가 뒤 도구의 입력으로 필요한 경우는 다음 절의 Agent 루프가 필요합니다.

In [ ]:
# 멀티 도구 호출 테스트
print("📊 멀티 도구 호출 테스트")
print("="*60)

multi_tool_questions = [
    "서울이랑 부산 날씨 비교해줘",
    "100달러가 환율 1350원일 때 얼마인지 계산하고, 서울 날씨도 알려줘"
]

for q in multi_tool_questions:
    run_tool_calling(q, tools, available_functions)

## 6️⃣ Agent 루프 — 결과를 보고 다시 판단한다

4️⃣의 `run_tool_calling` 은 도구를 **한 라운드** 실행하고 나면 무조건 최종 답을 만듭니다.
그래서 "서울 기온을 조회한 뒤 그 값에 3을 곱해줘" 처럼 **앞 도구의 결과가 있어야 다음 도구를 부를 수 있는** 요청은 처리하지 못합니다.

해결책은 단순합니다. 최종 응답 생성을 따로 두지 않고, **모델이 `tool_calls` 를 요청하는 동안 계속 도구를 실행해 돌려주고, 텍스트 응답이 오면 멈추는** 루프로 바꾸면 됩니다.

```python
while True:
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    if not msg.tool_calls:      # 최종 응답 → 루프 종료
        break

    # 1. 도구 호출 정보 추출 (msg.tool_calls)
    # 2. 실제 도구 실행
    # 3. 결과를 role="tool" 로 messages 에 추가
    # 4. 다음 반복으로 → 모델이 결과를 보고 추가 판단
```

이것이 Session 03에서 본 Claude Code 의 동작 구조이며, Session 05·06 에서도 그대로 재사용됩니다.

In [ ]:
# Agent 루프: 모델이 tool_calls 를 요청하는 동안 도구를 실행해 돌려주고, 텍스트 응답이 오면 종료
def run_agent(user_message, tools=tools, available_functions=available_functions,
              max_iterations=5, model=MODEL):
    """Tool Calling 을 반복해 다단계 요청을 처리하는 Agent 루프"""
    print(f"\n{'='*60}")
    print(f"👤 사용자: {user_message}")

    messages = [
        {"role": "system",
         "content": "당신은 도구를 활용해 정확한 정보를 제공하는 어시스턴트입니다. "
                    "앞 도구의 결과가 다음 도구의 입력으로 필요하면 순서대로 호출하세요."},
        {"role": "user", "content": user_message},
    ]

    for i in range(1, max_iterations + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=tools, tool_choice="auto",
        )
        msg = response.choices[0].message

        if not msg.tool_calls:                       # 도구 요청이 없으면 최종 응답
            print(f"\n🤖 최종 응답 (라운드 {i}): {msg.content}")
            return msg.content

        messages.append(msg)                         # 도구 호출이 담긴 assistant 메시지를 먼저 추가
        print(f"\n🔁 라운드 {i}: 도구 {len(msg.tool_calls)}개 호출")
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            if name in available_functions:
                result = available_functions[name](**args)
            else:
                result = {"error": f"Unknown function: {name}"}
            messages.append({                        # 결과를 tool_call_id 로 짝지어 돌려준다
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })

    print("⚠️ 최대 반복 횟수 도달 — 사람이 개입해야 합니다")
    return None

print("✅ run_agent 정의 완료")

In [ ]:
# 1회 왕복 vs Agent 루프 — 앞 도구의 결과가 다음 도구의 입력이 되는 요청
chained = "서울 현재 기온을 조회하고, 그 기온에 3을 곱한 값을 계산해줘"

print("▼ run_tool_calling (1회 왕복): calculate 를 부를 기회가 없어 모델이 암산으로 답한다 (도구 실행 로그에 🔢 가 없음)")
run_tool_calling(chained, tools, available_functions)

print("\n\n▼ run_agent (루프): 라운드 1 에서 get_weather → 라운드 2 에서 calculate")
run_agent(chained)

# 독립적인 멀티 도구도 루프 안에서 그대로 동작한다
_ = run_agent("서울이랑 부산 날씨 비교해줘")

### 1회 왕복과 Agent 루프의 차이

| | `run_tool_calling` (4️⃣) | `run_agent` (6️⃣) |
|---|---|---|
| 도구 라운드 | 최대 1회 | 모델이 멈출 때까지 반복 |
| 연쇄 요청 | 두 번째 도구를 부를 기회 없음 | 결과를 보고 다음 도구 호출 |
| 종료 조건 | 항상 2번째 호출에서 종료 | `tool_calls` 가 없을 때 |
| 안전장치 | 불필요 | `max_iterations` 로 무한 루프 방지 |

> **핵심**: Agent 는 새로운 API 가 아닙니다. 같은 Tool Calling 을 **루프 안에서** 부르는 것뿐입니다.
> 도구가 파일·터미널이면 Claude Code, 도구가 MCP 서버면 Session 05, 루프를 그래프로 그리면 Session 06 입니다.

## 7️⃣ 실전 도구 확장

도구를 늘리는 데 필요한 것은 **설명서 하나와 함수 하나**뿐입니다. `run_agent` 는 손대지 않습니다.

In [ ]:
# 실전 도구 세트 확장
print("📊 실전 Tool Calling 시나리오")
print("="*60)

# 실전용 도구 정의
advanced_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "주식 종목의 현재 가격을 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "종목 코드 (예: 005930, AAPL)"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "이메일을 발송합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string", "description": "수신자"},
                    "subject": {"type": "string", "description": "제목"},
                    "body": {"type": "string", "description": "본문"}
                },
                "required": ["to", "subject", "body"]
            }
        }
    }
]

# 시뮬레이션 함수
def get_stock_price(ticker):
    prices = {"005930": 72500, "AAPL": 195.50, "NVDA": 875.28}
    price = prices.get(ticker, random.randint(10000, 100000))
    result = {"ticker": ticker, "price": price, "change": f"+{random.uniform(-3, 5):.1f}%"}
    print(f"  💹 [도구 실행] get_stock_price(ticker='{ticker}') → {result}")
    return result

def send_email(to, subject, body):
    result = {"status": "sent", "to": to, "subject": subject}
    print(f"  📧 [도구 실행] send_email(to='{to}', subject='{subject}')")
    return result

# 기존 3개 + 실전 2개 통합 — run_agent 는 그대로
all_tools = tools + advanced_tools
all_functions = {
    **available_functions,
    "get_stock_price": get_stock_price,
    "send_email": send_email
}

print(f"✅ 총 {len(all_tools)}개 도구 정의 완료")
for tool in all_tools:
    print(f"  🔧 {tool['function']['name']}: {tool['function']['description'][:50]}")

In [ ]:
# 실전 시나리오 테스트 — Agent 루프로 실행
practical_questions = [
    "삼성전자 주가 알려줘",
    "김과장에게 회의 일정 변경 이메일 보내줘. 내일 3시로 변경됐어.",
    "서울 날씨 확인하고, 100달러가 환율 1350원일 때 얼마인지도 계산해줘",
    "고마워! 도움이 많이 됐어."  # 도구 불필요
]

for q in practical_questions:
    run_agent(q, all_tools, all_functions)

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| 도구 정의 | JSON Schema 로 작성한 **설명서** — 실제 함수와 이름으로 연결 |
| 첫 호출 | 모델은 실행하지 않고 `tool_calls` 로 **요청**만 한다 |
| 1회 왕복 파이프라인 | 판단 → 실행 → `role="tool"` 로 결과 전달 → 최종 응답 |
| 멀티 도구 | 독립적인 도구는 한 라운드에 동시 호출 |
| Agent 루프 | `tool_calls` 가 없을 때까지 반복 — 연쇄 요청 처리, `max_iterations` 로 안전장치 |

### 핵심 포인트

- 🎯 **도구 description 이 곧 모델의 판단 근거**입니다. 설명이 모호하면 잘못된 도구를 고릅니다.
- 🎯 **도구 실행은 언제나 우리 코드의 몫**입니다. 모델은 요청만 합니다.
- 🎯 **Agent = Tool Calling + 루프**. 새로운 API 가 아니라 호출 구조의 차이입니다.
- 🎯 도구 불필요한 질문에 도구를 부르지 않는 것도 중요합니다.

### 다음 노트북

- ➡️ **Session 05** (`05_mcp_agent.ipynb`): 이 노트북의 `get_weather`, `calculate`, `search_web` 을 MCP 서버로 분리합니다. `run_agent` 루프는 그대로 두고 **도구 목록 조회와 실행만 서버에 위임**합니다.
- ➡️ **Session 06** (`06_agent_tech_stack_langgraph.ipynb`): 이 루프를 LangGraph 그래프로 표현하고 멀티 에이전트로 확장합니다.